# Week 02 Day 05 — Generation Debug Lab

The Week-2 capstone notebook. Loads the `TinyGPT` stack (attention + block + decoding + KV cache)
and demonstrates the four **contracts** whose violations are this week's bug classes:

1. **Causal** — attention maps are strictly lower-triangular (no future leakage).
2. **Train/inference** — KV-cache generation is token-identical to the naive path (+ the decode speedup).
3. **Numeric** — a fully-masked attention row produces `NaN`; the fix keeps every row with ≥1 key.
4. **Decoding** — greedy / temperature / top-k / top-p / beam on the same prompt.

Companion regression suite: `../code/debug_lab.py` (the tests that would have caught all of it).

In [ ]:
import os, sys, math, time
import torch
import torch.nn.functional as F

# make the Week-2 code importable
CODE = os.path.abspath("../code")
if CODE not in sys.path:
    sys.path.insert(0, CODE)

from transformer_block import GPTConfig, TinyGPT
from decoding import generate, beam_search
from kv_cache import generate_cached, gpt_forward_cached, kv_cache_bytes
from debug_lab import _trained_successor, _random_model, _successor_batch

torch.manual_seed(0)

# Train a TinyGPT on the successor task (y = (x+1) mod V) so greedy should "count up".
model, cfg = _trained_successor(seed=0)
model.eval()
x, y = _successor_batch(cfg)
with torch.no_grad():
    final_ce = F.cross_entropy(model(x).reshape(-1, cfg.V), y.reshape(-1)).item()
print(f"cfg = {cfg}")
print(f"overfit CE = {final_ce:.4f}  (<< ln V = {math.log(cfg.V):.3f})  -> learning works")

## 1. Causal contract — attention maps are lower-triangular

`CausalSelfAttention.forward` doesn't return weights, so we recompute the softmax weights for one
block on a given input (mirroring its forward). A correct causal map has **zero mass above the
diagonal**: query row `i` only attends to keys `j <= i`. Any nonzero upper triangle == future leakage.

In [ ]:
@torch.no_grad()
def attention_weights(model, idx, layer=0):
    """Recompute the softmax attention weights (B,H,T,T) for one block, mirroring its forward."""
    B, T = idx.shape
    pos = torch.arange(T)
    h = model.tok_emb(idx) + model.pos_emb(pos)
    for i, blk in enumerate(model.blocks):
        if i == layer:
            attn = blk.attn
            xin = blk.norm1(h)
            q, k = attn._split(attn.Wq(xin)), attn._split(attn.Wk(xin))
            scores = (q @ k.transpose(-1, -2)) / math.sqrt(attn.d_k)
            causal = torch.tril(torch.ones(T, T, dtype=torch.bool))
            scores = scores.masked_fill(~causal, float("-inf"))
            return torch.softmax(scores, dim=-1)  # (B,H,T,T)
        h = blk(h)

idx = torch.randint(0, cfg.V, (1, cfg.T))
W = attention_weights(model, idx, layer=0)[0]  # (H, T, T)
upper = W.triu(diagonal=1).abs().max().item()
print(f"max attention mass ABOVE the diagonal = {upper:.2e}  (0 => causal, no leakage)")
print(f"every query row sums to 1? {torch.allclose(W.sum(-1), torch.ones_like(W.sum(-1)), atol=1e-6)}")

try:
    import matplotlib.pyplot as plt
    H = W.shape[0]
    fig, axes = plt.subplots(1, H, figsize=(3 * H, 3))
    for hh, ax in enumerate(axes):
        ax.imshow(W[hh], cmap="viridis", vmin=0, vmax=1)
        ax.set_title(f"head {hh}"); ax.set_xlabel("key j"); ax.set_ylabel("query i")
    fig.suptitle("Layer-0 causal attention (lower-triangular = correct)")
    plt.tight_layout(); plt.show()
except ImportError:
    print("(matplotlib not installed — showing head 0 numerically)")
    print(W[0].round(decimals=2))

## 2. Decoding contract — greedy / temperature / top-k / top-p / beam

Same prompt, five strategies. On this **low-entropy** successor task, the search methods (greedy, beam)
correctly "count up"; sampling adds controlled diversity. On open-ended text, greedy/beam is exactly
where you'd see the repetition-attractor degeneration.

In [ ]:
prompt = torch.tensor([[3, 4, 5]])
N = 5

def cont(out):  # strip the prompt, show only the generated continuation
    return out[0, prompt.shape[1]:].tolist()

g = torch.Generator().manual_seed(0)
print("greedy         :", cont(generate(model, prompt, N, greedy=True)))
print("temp=0.7       :", cont(generate(model, prompt, N, temperature=0.7, generator=g)))
print("temp=1.5       :", cont(generate(model, prompt, N, temperature=1.5, generator=g)))
print("top-k=3        :", cont(generate(model, prompt, N, temperature=1.0, top_k=3, generator=g)))
print("top-p=0.9      :", cont(generate(model, prompt, N, temperature=1.0, top_p=0.9, generator=g)))
print("beam (w=3)     :", beam_search(model, prompt, beam_width=3, max_new_tokens=N,
                                      length_penalty=1.0)[0, prompt.shape[1]:].tolist())
print("\nexpected count-up:", [(5 + i + 1) % cfg.V for i in range(N)])

## 3. Train/inference contract — KV cache is token-identical (+ speedup)

The cache is an optimization, so it must not change a single token vs. the naive path. We assert
token-identical greedy output, then time both paths to show the decode speedup. (We use a **random**
model for the equality check — its logits depend on all positions via attention, so a broken mask
would actually change tokens; the successor model is insensitive to attention bugs.)

In [ ]:
# --- equality (small random model) ---
rnd = _random_model(seed=1); rnd.eval()
p = torch.randint(0, rnd.cfg.V, (1, 3))
naive  = generate(rnd, p, max_new_tokens=5, greedy=True)
cached = generate_cached(rnd, p, max_new_tokens=5, greedy=True)
print("cached == uncached (token-identical)?", torch.equal(naive, cached))

# --- speedup (bigger context so O(N^2) vs O(N^3) shows) ---
big = TinyGPT(GPTConfig(V=64, d=128, n_heads=4, n_layers=4, T=256)); big.eval()
p2, N2 = torch.randint(0, 64, (1, 4)), 200

t0 = time.perf_counter(); _ = generate(big, p2, max_new_tokens=N2, greedy=True); t_naive = time.perf_counter() - t0
t0 = time.perf_counter(); _ = generate_cached(big, p2, max_new_tokens=N2, greedy=True); t_cached = time.perf_counter() - t0
print(f"naive  generate: {t_naive*1e3:7.1f} ms")
print(f"cached generate: {t_cached*1e3:7.1f} ms   ->  {t_naive/t_cached:.2f}x faster over {N2} tokens")

# --- the memory math this optimization trades off ---
b = kv_cache_bytes(GPTConfig(V=32000, d=4096, n_heads=32, n_layers=32, T=4096), seq_len=4096, batch=1)
print(f"\n7B fp16 KV cache @ 4k, B=1: {b/2**30:.2f} GiB  ({b/4096/2**10:.0f} KiB/token)")

## 4. Numeric contract — the fully-masked row NaN, and its fix

The single most common attention NaN: a query row that is **entirely `-inf`** before softmax
(e.g. a padding mask that masks every key, or a bad offset). `softmax` over all `-inf` is `0/0 → NaN`,
and the NaN then poisons the whole forward. The fix is to **guarantee every row keeps ≥1 key**
(a causal row always keeps itself) — which is exactly what `test_masked_row_not_all_inf` enforces.

In [ ]:
# REPRO: row 0 attends to NOTHING (all -inf) -> NaN
scores = torch.randn(4, 4)
mask = torch.tril(torch.ones(4, 4, dtype=torch.bool))
mask[0] = False                      # <- the bug: nothing is visible to query 0
bad = torch.softmax(scores.masked_fill(~mask, float("-inf")), dim=-1)
print("has NaN?", torch.isnan(bad).any().item(), " row 0 =", bad[0].tolist())

# FIX: a valid causal mask always keeps the diagonal (query i sees key i) -> no empty row
mask_ok = torch.tril(torch.ones(4, 4, dtype=torch.bool))
good = torch.softmax(scores.masked_fill(~mask_ok, float("-inf")), dim=-1)
print("has NaN?", torch.isnan(good).any().item(), " every row keeps >=1 key:",
      mask_ok.any(-1).all().item())